In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

d:\code\FSFM_Lite_Project


In [2]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc
)

import matplotlib.pyplot as plt
import seaborn as sns

from torchvision import transforms
from torch.utils.data import DataLoader

from src.datasets.celeba_spoof_dataset import (CelebASpoofDataset)
from src.models.fsfm_lite import (FSFMLite)

In [ ]:
PROJECT_ROOT = Path("/kaggle/input/datasets/thanhluanchimto/source-code")
METADATA_ROOT = Path("/kaggle/input/datasets/thanhluanchimto/meta-data/metadata")
DATA_ROOT = Path("/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof")
CHECKPOINT_ROOT = Path("/kaggle/working/checkpoints")
REPORT_ROOT = Path("/kaggle/working/reports")
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.append(str(PROJECT_ROOT))

In [ ]:
test_df = pd.read_csv(METADATA_ROOT / "test_df.csv")
print(test_df.shape)
test_df.head()

In [ ]:
face_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
test_dataset = CelebASpoofDataset(test_df, transform=face_transform)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

print(len(test_dataset))

In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = FSFMLite()

checkpoint = torch.load(
    "./outputs/checkpoints/best_fsfm_lite.pth",
    map_location=device
)

model.load_state_dict(checkpoint)
model = model.to(device)
model.eval()

print("Model Loaded")

In [ ]:
all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():

    for batch in tqdm(test_loader):

        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        logits = model(images)

        probs = torch.softmax(logits, dim=1)[:,1]
        preds = logits.argmax(dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

In [ ]:
acc = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

In [ ]:
report = classification_report(all_labels, all_preds, target_names=["Live", "Spoof"])

print(report)

with open(
    REPORT_ROOT /
    "classification_report.txt",
    "w"
) as f:

    f.write(report)

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.savefig(REPORT_ROOT / "confusion_matrix.png")

plt.show()

In [ ]:
fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8,6))
plt.plot(fpr, tpr,label=f"AUC={roc_auc:.4f}")
plt.plot([0,1], [0,1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.savefig(REPORT_ROOT / "roc_curve.png")

plt.show()

print(f"AUC: {roc_auc:.4f}")

In [ ]:
metrics = {
    "accuracy": float(acc),
    "precision": float(precision),
    "recall": float(recall),
    "f1_score": float(f1),
    "auc": float(roc_auc)
}

import json

with open(
    REPORT_ROOT /
    "metrics.json",
    "w"
) as f:

    json.dump(
        metrics,
        f,
        indent=4
    )

metrics